$$\Large\boxed{\text{AME 5202 Deep Learning, Even Semester 2026}}$$

$$\large\text{Theme}: \underline{\text{understanding hooks in PyTorch}}$$

---

Load essential libraries

---

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
plt.style.use('dark_background')
%matplotlib inline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

---

**The patient data matrix with output labels and initial weight matrix**

![patient dataset](https://1drv.ms/i/s!AjTcbXuSD3I3hspfrgklysOtJMOjaA?embed=1&width=800)

---



In [ ]:
# Patients data matrix
X = torch.tensor([[72, 120, 37.3, 104, 32.5],
                 [85, 130, 37.0, 110, 14],
                 [68, 110, 38.5, 125, 34],
                 [90, 140, 38.0, 130, 26],
                 [84, 132, 38.3, 146, 30],
                 [78, 128, 37.2, 102, 12]], dtype = torch.float64)
print(f'Patient data matrix X:\n {X}') #f-string in Python
print('---------')

# Initial Weights matrix (trainable tensor)
W = torch.tensor([[-0.1, 0.5, 0.3],
                  [0.9, 0.3, 0.5],
                  [-1.5, 0.4, 0.1],
                  [0.1, 0.1, -1.0],
                  [-1.2, 0.5, -0.8]], dtype = torch.float64,
                 requires_grad = True)
print(f'Initial weights matrix:\n {W}')
print('---------')

# Create a 1D-numpy array of output labels (equivalent to a rank-1 tensor in
# PyTorch which itself is equivalent to a vector in pen & paper)
y = np.array(['non-diabetic',
              'diabetic',
              'non-diabetic',
              'pre-diabetic',
              'diabetic',
              'pre-diabetic'])
# Creating a one-hot encoder object
ohe = OneHotEncoder(sparse_output = False)
# Create the one-hot encoded true output labels matrix
Y = torch.tensor(ohe.fit_transform(y.reshape(-1, 1)), dtype = torch.float64)
print(f'One-hot encoded output labels matrix:\n {Y}')
print('---------')

# Standardize the data
sc = StandardScaler() # create a standard scaler object
X_std = torch.tensor(sc.fit_transform(X), dtype = torch.float64)
print(f'The standardized data matrix:\n{X_std}')

---

Softmax classifier module

---

In [ ]:
class SoftmaxNetwork(nn.Module):
    def __init__(self, W_init):
      super().__init__()
      self.linear = nn.Linear(5, 3, bias = False, dtype = torch.float64)
        
      # Manually set initial weights
      with torch.no_grad():
        self.linear.weight.copy_(W_init.T)

    def forward(self, x):
      logits = self.linear(x)
      probs = F.softmax(logits, dim = -1)
      return probs

---

Instantiate model

---

In [ ]:
model = SoftmaxNetwork(W)

---

Hooks in PyTorch are a way to attach a function to a model so that it automatically runs at a specific moment during training or inference. A hook can be attached to:

- Forward pass → see or modify activations
- Backward pass → see or modify gradients

Now we will attach a hook to the linear layer, so we can see the raw scores (logits) before softmax.

---

In [ ]:
forward_storage = {}

def forward_hook(module, input, output):
  print("Forward Hook Triggered")
  print(f"Logits:\n {output}")
  forward_storage["logits"] = output.detach()

forward_handle = model.linear.register_forward_hook(forward_hook)

---

Now we will attach a backward hook to inspect gradients

---

In [ ]:
def backward_hook(module, grad_input, grad_output):
  print("Backward Hook Triggered")
  print(f"Gradient w.r.t logits:\n{grad_output[0]}")

backward_handle = model.linear.register_full_backward_hook(backward_hook)

---

One training step.

---

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# Forward pass
probs = model(X_std)

# Cross-entropy loss (manual form for one-hot labels)
loss = torch.mean(-torch.sum(Y * torch.log(probs)))
print(f"Loss: {loss.item()}")

# Backward pass
optimizer.zero_grad()
loss.backward()

# Update
optimizer.step()